# Benchmark: Critique-Revise vs LLM-as-a-Judge

This offline #129 example compares two scripted arms on three synthetic tasks.
Each arm receives the same frozen starting draft. Critique-Revise applies one
scripted revision and returns the resulting artifact. The judge returns the
original draft with an approve or flag decision for human review.

The independent rubric scores the artifact returned by each arm, even when
the judge flags it. A flag is recorded separately and earns no rubric points.
The fixture's scripted behaviors illustrate reporting. Live quality, latency,
token usage, and cost are unmeasured.

In [ ]:
import hashlib
import json
from pathlib import Path

FIXTURE_NAME = "benchmark_critique_vs_judge_fixture.json"
cwd = Path.cwd().resolve()
fixture_path = next(
    (path for parent in (cwd, *cwd.parents)
     for path in (parent / "examples" / FIXTURE_NAME, parent / FIXTURE_NAME)
     if path.is_file()),
    None,
)
if fixture_path is None:
    raise FileNotFoundError(f"{FIXTURE_NAME} was not found in the current directory or checkout.")

fixture = json.loads(fixture_path.read_text(encoding="utf-8"))
canonical_fixture = json.dumps(fixture, sort_keys=True, separators=(",", ":")).encode("utf-8")
fixture_sha256 = hashlib.sha256(canonical_fixture).hexdigest()

In [ ]:
def score_artifact(artifact, expected_fields):
    """Score a structured artifact using only the independently specified checks."""
    return {
        field: int(
            field in artifact
            and type(artifact[field]) is type(expected)
            and artifact[field] == expected
        )
        for field, expected in expected_fields.items()
    }


rows = []
for task in fixture["tasks"]:
    starting_draft = task["starting_draft"]
    critique = task["critique_revise"]
    judge = task["llm_judge"]
    expected = fixture["rubric"]["expected_by_task"][task["id"]]
    critique_artifact = {**starting_draft, **critique["revision"]}
    judge_artifact = dict(starting_draft)
    rows.append({
        "task_id": task["id"],
        "prompt": task["prompt"],
        "starting_draft": starting_draft,
        "draft_points": sum(score_artifact(starting_draft, expected).values()),
        "possible_points": len(expected),
        "critique_revise": {
            "artifact": critique_artifact,
            "feedback": critique["feedback"],
            "checks": score_artifact(critique_artifact, expected),
        },
        "llm_judge": {
            "artifact": judge_artifact,
            "decision": judge["decision"],
            "feedback": judge["feedback"],
            "checks": score_artifact(judge_artifact, expected),
        },
    })

rows

In [ ]:
arm_names = {"critique_revise": "Critique-Revise", "llm_judge": "LLM-as-a-Judge"}
possible_points = sum(row["possible_points"] for row in rows)
summary = {
    "fixture_version": fixture["fixture_version"],
    "fixture_sha256": fixture_sha256,
    "mode": "stub",
    "task_count": len(rows),
    "starting_draft_points": sum(row["draft_points"] for row in rows),
    "possible_points": possible_points,
    "arms": {
        arm: {
            "points": sum(sum(row[arm]["checks"].values()) for row in rows),
            "latency_seconds": None,
            "input_tokens": None,
            "output_tokens": None,
            "cost_usd": None,
        }
        for arm in arm_names
    },
    "judge_flagged_tasks": sum(row["llm_judge"]["decision"] == "flag" for row in rows),
}

lines = [
    f"Offline stub fixture {summary['fixture_version']} (SHA-256 `{fixture_sha256}`)",
    "",
    fixture["rubric"]["description"],
    "",
    "| Task | Shared draft | Critique-Revise artifact | Judge artifact | Judge decision |",
    "| --- | ---: | ---: | ---: | --- |",
]
for row in rows:
    possible = row["possible_points"]
    scores = [sum(row[arm]["checks"].values()) for arm in arm_names]
    lines.append(
        f"| {row['task_id']} | {row['draft_points']}/{possible} | "
        f"{scores[0]}/{possible} | {scores[1]}/{possible} | {row['llm_judge']['decision']} |"
    )
lines.extend([
    "",
    "| Arm | Synthetic rubric points | Latency | Input tokens | Output tokens | Cost |",
    "| --- | ---: | --- | --- | --- | --- |",
])
for arm, title in arm_names.items():
    result = summary["arms"][arm]
    metrics = [
        "Unknown" if result[key] is None else str(result[key])
        for key in ("latency_seconds", "input_tokens", "output_tokens", "cost_usd")
    ]
    lines.append(f"| {title} | {result['points']}/{possible_points} | " + " | ".join(metrics) + " |")
lines.extend([
    "",
    f"Judge flagged {summary['judge_flagged_tasks']}/{len(rows)} drafts.",
    "Scores describe scripted artifacts in this fixture. Live performance is unmeasured.",
])
MARKDOWN_TABLE = "\n".join(lines)
print(MARKDOWN_TABLE)
summary